# Diffusion-ES + AnchorWM: where the expert does better

Why does fine-tuning the scorer barely move closed-loop success? Look at scenes the logged expert drives cleanly (log replay) but the planner fails, and at every replan ask: **was a safe plan in the ES pool?**

* safe plan in the final pool, unsafe one chosen -> the **scorer** ranks wrong (what fine-tuning can fix)
* safe plan only in the initial pool -> ES dropped it
* no safe plan at all -> a **proposal** problem (the diffusion policy / ES), which no scorer fine-tuning fixes

1. Candidate scenes from the closed-loop results (base vs fine-tuned vs expert).
2. Re-run one scene with per-replan dumps (`anchor_wm/slurm/babel/es_dumps.sh`; needs a GPU, or submit it with sbatch).
3. Video of the executed plan and the expert, per-replan table, and per-replan panels: ES pools with scores against log-replay labels, and the decoded AnchorWM ego anchors.

Run on a babel compute node (`/data` is not mounted on the login node) with the `waymax_rs_ishan` kernel.

In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
import ipywidgets as w
from IPython.display import HTML, display
import anchor_wm.viz_failures as vf

pd.set_option("display.max_rows", 200)

## 1. Configuration

In [ ]:
W = "/data/user_data/mineuih/ishan/world_model_sim"
RUNS = {   # closed-loop runs on valfail1568 (same scenes, seed 0, log agents); the first two are compared below
    "base": f"{W}/es_awm_v2_scratch_joint_current_fix55k_final_log_r10_seed0_valfail1568",
    "ft":   f"{W}/es_awm_ftset_v2current_current_best_log_r10_seed0_valfail1568",
}
EXPERT = f"{W}/log_replay_valfail1568"
DUMP_ROOT = "/data/user_data/mineuih/ishan/failure_dumps"   # <DUMP_ROOT>/<run>/<scene>/ per re-run
BASE, FT = list(RUNS)[:2]

## 2. Candidate scenes

Outcome per scene: `success`, `overlap`, `offroad` (first failure) or `goal_miss`; `<run>_t` is the failure time in s. Only scenes the expert drives cleanly count: elsewhere the label or the map is broken, not the planner.

In [ ]:
df = vf.scene_table(RUNS, EXPERT)
clean = df[df["expert"] == "success"]
print(f"{len(df)} scenes, expert clean on {len(clean)}")
display(pd.DataFrame({r: clean[r].value_counts() for r in RUNS}).fillna(0).astype(int))
# base -> fine-tuned on the expert-clean scenes: off-diagonal = what fine-tuning fixed / broke
display(pd.crosstab(clean[BASE], clean[FT], rownames=[BASE], colnames=[FT], margins=True))

In [ ]:
# Where the expert does better than the fine-tuned planner, by failure type
cand = clean[clean[FT] != "success"].sort_values([FT, f"{FT}_t"])
display(cand[FT].value_counts())
cand.drop(columns="key")

## 3. Re-run one scene with per-replan dumps

The re-run uses one world per batch (the closed-loop runs used 5), so its sampling differs and its outcome can differ from the table; the outcome printed below is the re-run's own. A finished re-run is reused.

In [ ]:
SCENE = cand.index[0]   # <- pick a scene from the table
RUN = FT                # <- which run's checkpoints to re-run
out = f"{DUMP_ROOT}/{RUN}/{SCENE}"
display(df.loc[[SCENE]].drop(columns="key"))

In [ ]:
if not vf.load_results(out):                                         # no finished re-run yet
    ckpt, scorer_ckpt = vf.run_checkpoints(RUNS[RUN])
    vf.write_targets([df.loc[SCENE, "key"]], f"{out}/targets.json")
    env = dict(os.environ, CKPT=ckpt, SCORER_CKPT=scorer_ckpt, TARGETS=f"{out}/targets.json", OUT=out, ROOT=str(ROOT))
    script = str(ROOT / "anchor_wm/slurm/babel/es_dumps.sh")
    if os.environ.get("CUDA_VISIBLE_DEVICES"):
        subprocess.run(["bash", script], env=env, check=True)        # ~2-3 min per scene
    else:                                                            # no GPU here: submit, then re-run this cell
        print(subprocess.run(["sbatch", script], env=env, capture_output=True, text=True).stdout)
scene = vf.load_scene(out)
print("re-run outcome:", scene["failure"])

## 4. Executed plan vs the expert

Blue box and black path: the executed ego; dashed grey box and line: the logged expert; orange: the plan chosen at the latest replan.

In [ ]:
HTML(vf.animate(scene).to_jshtml())

## 5. Every replan

* `chosen` / `expert`: log-replay label of the chosen plan and of the expert path moved onto the ego (`expert_offset_m` from the log; large offsets make the expert comparison less meaningful)
* `expert_score`, `expert_rank`: the learned score of that expert path and its rank in the final pool (1 = would be chosen)
* `safe_final` / `safe_init`: safe plans in the final / initial pool, `best_safe_score` the highest-scored one
* `chosen_goal_m` / `expert_goal_m`: distance from the plan end to the goal (goal misses)
* `verdict`: `scorer` (a safe plan was there), `ES dropped safe`, `no safe proposal`, or `ok`

In [ ]:
tab = vf.replan_table(scene)
display(tab)
display(tab["verdict"].value_counts())

Left: final ES pool coloured by learned score, the chosen plan, the expert moved onto the ego and the executed second. Middle: every initial / final candidate's score with its log-replay label (chosen ringed, expert dashed). Right: AnchorWM decoded ego anchors.

In [ ]:
times = sorted(scene["steps"])
w.interact(lambda t: vf.replan_figure(scene, t),
           t=w.SelectionSlider(options=[(f"{x / 10:.0f} s", x) for x in times], value=times[0], description="replan"));